In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("/content/appointments_clean.csv")
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


##Task 3.1 — Features, split and baseline

In [3]:
from sklearn.model_selection import train_test_split

feature_cols = ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel', 'visit_type']
X = df[feature_cols].copy()

X['reminder_sent'] = X['reminder_sent'].map({'Yes': 1, 'No': 0})

X = pd.get_dummies(X, columns=['booking_channel', 'visit_type'])
y = df['no_show']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Report statistics
print(f"Training set size (X_train): {X_train.shape}")
print(f"Test set size (X_test): {X_test.shape}")
print(f"Number of no-shows in test set: {y_test.sum()}")

# Baseline accuracy of always predicting 'Attended' (no_show = 0)
baseline_accuracy = (y_test == 0).mean()
print(f"Baseline accuracy (always predicting 'Attended'): {baseline_accuracy:.4f}")

Training set size (X_train): (792, 11)
Test set size (X_test): (198, 11)
Number of no-shows in test set: 49
Baseline accuracy (always predicting 'Attended'): 0.7525


##Task 3.2 — Logistic regression

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import numpy as np
import pandas as pd

# 1. Scaling
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 2. Fit Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

# 3. Predict
y_pred = model.predict(X_test_s)
y_pred_proba = model.predict_proba(X_test_s)[:, 1]

# 4. Report metrics
print("=== Performance Metrics on Test Set ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")

# 5. Coefficients and exponentiated coefficients
coefficients = model.coef_[0]
features = X.columns

print("=== Coefficients per Feature ===")
for feat, coef in zip(features, coefficients):
    print(f"{feat}: {coef:.4f}")

# Get index and scaler scale for reminder_sent
reminder_idx = list(features).index('reminder_sent')
reminder_coef_scaled = coefficients[reminder_idx]
reminder_std = scaler.scale_[reminder_idx]

# Unscaled coefficient (divided by standard deviation)
reminder_coef_unscaled = reminder_coef_scaled / reminder_std

exp_scaled = np.exp(reminder_coef_scaled)
exp_unscaled = np.exp(reminder_coef_unscaled)
### Odd Ratios for Reminder Sent

print(
"""
python
print(f"\n=== Odds Ratios for reminder_sent ===")
print(f"Scaled coefficient (exp): {exp_scaled:.4f}")
print(f"Unscaled coefficient (exp): {exp_unscaled:.4f}")


-- Explanation

The two values are different because they represent different things. The scaled coefficient shows how the odds change when `reminder_sent` increases by one standard deviation, while the unscaled coefficient shows the effect of going from 0 (No reminder) to 1 (Reminder sent).

-- Conclusion

The model suggests that sending a reminder is associated with a lower chance of a patient missing their appointment. The odds ratio is approximately **0.54**, which means that patients who receive a reminder have about **46% lower odds of missing their appointment**, compared with those who do not receive one, assuming all other features remain constant.

"""
)

=== Performance Metrics on Test Set ===
Accuracy: 0.7879
Confusion Matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC Score: 0.7889

=== Coefficients per Feature ===
age: -0.2539
distance_km: 0.2534
fee_inr: 0.4528
lead_days: 0.6145
prior_no_shows: 0.6496
reminder_sent: -0.5807
booking_channel_App: -0.0468
booking_channel_Phone: 0.2159
booking_channel_Walk-In: -0.1864
visit_type_Consultation: 0.0157
visit_type_Follow-up: -0.0157

python
print(f"
=== Odds Ratios for reminder_sent ===")
print(f"Scaled coefficient (exp): {exp_scaled:.4f}")
print(f"Unscaled coefficient (exp): {exp_unscaled:.4f}")


-- Explanation

The two values are different because they represent different things. The scaled coefficient shows how the odds change when `reminder_sent` increases by one standard deviation, while the unscaled coefficient shows the effect of going from 0 (No reminder) to 1 (Reminder sent).

-- Conclusion

The model suggests that sending a reminder is asso

##Task 3.3 — Decision tree and overfitting

In [6]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state = 42,
)

dt.fit(X_train_s,y_train)
y_pred = dt.predict(X_test_s)
y_t_pred = dt.predict(X_train_s)

print("For First Model")

print(accuracy_score(y_test, y_pred))
print(accuracy_score(y_train, y_t_pred))

dt2 = DecisionTreeClassifier(
    random_state = 42,
    max_depth = 3
)

dt2.fit(X_train_s,y_train)
y_pred2 = dt2.predict(X_test_s)
y_t_pred2 = dt2.predict(X_train_s)

print("For Second Model")
print(accuracy_score(y_test, y_pred2))
print(accuracy_score(y_train, y_t_pred2))

print("""
-- Reason
  Reason of rhe change in accuracy both traing set and test set is the depth of the Decision tree it meomrize the traing data that good so that give 100% accuracy but in unseen data it give only 67% accuracy,
  the major change happen because of the depth of the tree.
""")

For First Model
0.6717171717171717
1.0
For Second Model
0.7777777777777778
0.821969696969697

-- Reason
  Reason of rhe change in accuracy both traing set and test set is the depth of the Decision tree it meomrize the traing data that good so that give 100% accuracy but in unseen data it give only 67% accuracy,
  the major change happen because of the depth of the tree.



##Task 3.4 — Random forest and a comparison table

In [8]:
from sklearn.ensemble import RandomForestClassifier

# 1. Fit Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_s, y_train)

# 2. Evaluate Random Forest
y_pred_rf = rf.predict(X_test_s)
y_pred_proba_rf = rf.predict_proba(X_test_s)[:, 1]

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_roc_auc = roc_auc_score(y_test, y_pred_proba_rf)

print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"ROC-AUC:   {rf_roc_auc:.4f}\n")

# 3. Feature Importances
importances = rf.feature_importances_
features = X.columns
indices = np.argsort(importances)[::-1]

for i in range(3):
    print(f"{i+1}. {features[indices[i]]}: {importances[indices[i]]:.4f}")
print()


y_pred_dt2 = dt2.predict(X_test_s)
y_pred_proba_dt2 = dt2.predict_proba(X_test_s)[:, 1]
dt2_accuracy = accuracy_score(y_test, y_pred_dt2)
dt2_recall = recall_score(y_test, y_pred_dt2)
dt2_roc_auc = roc_auc_score(y_test, y_pred_proba_dt2)

y_pred_lr = model.predict(X_test_s)
y_pred_proba_lr = model.predict_proba(X_test_s)[:, 1]
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_roc_auc = roc_auc_score(y_test, y_pred_proba_lr)

metrics_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree (Depth 3)', 'Random Forest'],
    'Accuracy': [lr_accuracy, dt2_accuracy, rf_accuracy],
    'Recall': [lr_recall, dt2_recall, rf_recall],
    'ROC-AUC': [lr_roc_auc, dt2_roc_auc, rf_roc_auc]
})

print(metrics_df.to_string(index=False))

print("""
=== Model Comparison Discussion ===
Based on the table, Logistic Regression currently appears to be the best-performing model overall.
- It achieves the highest ROC-AUC (~0.789), reflecting its strong general capability to distinguish between those who show up and those who do not.
- It also exhibits the highest Recall (~0.347) among the three models, which is particularly valuable since the primary cost/concern is typically failing to identify patients who will be no-shows (false negatives).
- While the Random Forest model achieves slightly better raw Accuracy (~0.793), its recall (~0.224) and ROC-AUC (~0.748) are lower, suggesting that it might be slightly overfitting or less robust on the test set compared to the linear relationship found by Logistic Regression.
""")

Accuracy:  0.7475
Recall:    0.3061
ROC-AUC:   0.7077

1. lead_days: 0.2199
2. age: 0.1968
3. distance_km: 0.1947

                  Model  Accuracy   Recall  ROC-AUC
    Logistic Regression  0.787879 0.346939 0.788933
Decision Tree (Depth 3)  0.777778 0.224490 0.699699
          Random Forest  0.747475 0.306122 0.707711

=== Model Comparison Discussion ===
Based on the table, Logistic Regression currently appears to be the best-performing model overall. 
- It achieves the highest ROC-AUC (~0.789), reflecting its strong general capability to distinguish between those who show up and those who do not.
- It also exhibits the highest Recall (~0.347) among the three models, which is particularly valuable since the primary cost/concern is typically failing to identify patients who will be no-shows (false negatives).
- While the Random Forest model achieves slightly better raw Accuracy (~0.793), its recall (~0.224) and ROC-AUC (~0.748) are lower, suggesting that it might be slightly overfitt

##Task 3.5 — Tune, choose the model and the threshold, then test once

In [13]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid={"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc"
)
# Tuning is performed on the scaled training features
rf_grid.fit(X_train_s, y_train)

print("=== (a) Random Forest Grid Search Results ===")
print(f"Best Parameters: {rf_grid.best_params_}")
print(f"Best CV ROC-AUC: {rf_grid.best_score_:.4f}\n")

# --- (b) Choose Final Model via 5-Fold Cross-Validation ---
# Setup pipelines to prevent data leakage during CV
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, random_state=42))])
dt_pipe = Pipeline([("scaler", StandardScaler()), ("model", DecisionTreeClassifier(max_depth=3, random_state=42))])
rf_pipe = Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
tuned_rf_pipe = Pipeline([("scaler", StandardScaler()), ("model", rf_grid.best_estimator_)])

cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_dt = cross_val_score(dt_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_tuned_rf = cross_val_score(tuned_rf_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()

print("=== (b) 5-Fold Cross-Validated ROC-AUC Comparison ===")
print(f"Logistic Regression CV ROC-AUC:      {cv_auc_lr:.4f}")
print(f"Decision Tree (depth-3) CV ROC-AUC:  {cv_auc_dt:.4f}")
print(f"Default Random Forest CV ROC-AUC:    {cv_auc_rf:.4f}")
print(f"Tuned Random Forest CV ROC-AUC:      {cv_auc_tuned_rf:.4f}\n")

=== (a) Random Forest Grid Search Results ===
Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7669

=== (b) 5-Fold Cross-Validated ROC-AUC Comparison ===
Logistic Regression CV ROC-AUC:      0.7737
Decision Tree (depth-3) CV ROC-AUC:  0.6783
Default Random Forest CV ROC-AUC:    0.7450
Tuned Random Forest CV ROC-AUC:      0.7672



In [17]:
winner_pipe = lr_pipe
oof_probs = cross_val_predict(winner_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

def evaluate_threshold(probs, y_true, threshold):
    preds = (probs >= threshold).astype(int)
    precision = precision_score(y_true, preds)
    recall = recall_score(y_true, preds)
    f1 = f1_score(y_true, preds)
    flagged = preds.sum()
    return precision, recall, f1, flagged

for thresh in [0.5, 0.35]:
    prec, rec, f1, flag = evaluate_threshold(oof_probs, y_train, thresh)
    print(f"Threshold {thresh}: Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}, Flagged={flag}")

print("\nDecision rationale: The threshold 0.35 yields a much higher Recall (more flagged potential no-shows) and a higher F1 score compared to 0.5.")
print("Recall is highly critical because a missed warning (false negative) results in a costly unutilized slot, whereas an extra follow-up call (false positive) only costs a minute of staff time.")

Threshold 0.5: Precision=0.5753, Recall=0.2485, F1=0.3471, Flagged=73
Threshold 0.35: Precision=0.5130, Recall=0.4675, F1=0.4892, Flagged=154

Decision rationale: The threshold 0.35 yields a much higher Recall (more flagged potential no-shows) and a higher F1 score compared to 0.5.
Recall is highly critical because a missed warning (false negative) results in a costly unutilized slot, whereas an extra follow-up call (false positive) only costs a minute of staff time.


In [18]:
winner_pipe.fit(X_train, y_train)
test_probs = winner_pipe.predict_proba(X_test)[:, 1]
final_threshold = 0.35
test_preds = (test_probs >= final_threshold).astype(int)

test_precision = precision_score(y_test, test_preds)
test_recall = recall_score(y_test, test_preds)
test_flagged = test_preds.sum()
test_auc = roc_auc_score(y_test, test_probs)

print(f"Precision:      {test_precision:.4f}")
print(f"Recall:         {test_recall:.4f}")
print(f"Flagged count:  {test_flagged}")
print(f"ROC-AUC Score:  {test_auc:.4f}\n")

Precision:      0.5714
Recall:         0.5714
Flagged count:  49
ROC-AUC Score:  0.7889



In [20]:
test_indices = X_test.index
appointment_ids = df.loc[test_indices, 'appointment_id']

risk_scores_df = pd.DataFrame({
    'appointment_id': appointment_ids,
    'no_show': y_test,
    'risk_prob': test_probs,
    'flag': test_preds
})

risk_scores_df.to_csv('test_risk_scores.csv', index=False)
print(risk_scores_df.head())

print("\nThreshold Choice Sentence: In reality, a clinic would choose this threshold by balancing the daily call capacity of the front desk against the financial losses of empty clinical slots.")

    appointment_id  no_show  risk_prob  flag
213          A0217        0   0.021565     0
331          A0336        0   0.150917     0
501          A0508        0   0.518692     1
309          A0314        0   0.191300     0
88           A0089        0   0.167718     0

Threshold Choice Sentence: In reality, a clinic would choose this threshold by balancing the daily call capacity of the front desk against the financial losses of empty clinical slots.
